# Redis: Guide for Python (`redis-py`)

This notebook is a **hands-on guide** to Redis from Python using **`redis`** (redis-py). It assumes you already know Redis conceptually and want a **structured reference** with runnable examples.

**Requirements:** a running Redis server (default `localhost:6379`) and `pip install redis`.

If Redis is not available, code cells **catch errors** and print a short message so the notebook still opens cleanly.


### Contents

- [1. Basic key–value operations](#1-basic-keyvalue-operations)
- [2. Core Redis data types (lists, hashes, sets, sorted sets, …)](#2-core-redis-data-types)
- [3. PUB/SUB in one notebook (threads)](#3-pubsub-in-one-notebook)
- [4. Keyspace notifications](#4-keyspace-notifications)
- [5. Streams: fundamentals (`XADD`, `XREAD`, `XRANGE`)](#5-streams-fundamentals)
- [6. Streams: consumer groups (`XGROUP`, `XREADGROUP`, `XACK`)](#6-streams-consumer-groups)
- [7. Streams: multi-consumer patterns, pending, and recovery](#7-streams-multi-consumer-patterns)
- [8. Streams: persistence semantics, trimming, and memory](#8-streams-persistence-and-trimming)
- [9. Pipelines, transactions (`MULTI`/`EXEC`), and Lua](#9-pipelines-transactions-and-lua)
- [10. Server-side persistence (RDB/AOF) and replication (overview)](#10-rdb-aof-and-replication)
- [11. Practical checklist](#11-practical-checklist)
- [12. Companion: trading and low-latency patterns](#12-companion-trading-and-low-latency-patterns)


In [124]:
REDIS_ARGS = dict(host = "localhost", db = 0,
  decode_responses = True, password = input("REDIS PASSWORD"))
REDIS_PREFIX = "TRAINING:"

## 1. Basic key–value operations

Redis is often used as a **key–value store** with optional **TTL** and **atomic** updates. Prefer **`SCAN`** over **`KEYS *`** in production (non-blocking, cursor-based).

Common primitives: **`GET`/`SET`**, **`SET`** with **`NX`**/**`XX`**, **`DEL`**, **`EXPIRE`/`TTL`**, **`INCR`/`DECR`**, **`MGET`/`MSET`**, **`SCAN`**.


In [2]:
from redis import Redis

redis = Redis(**REDIS_ARGS)
redis.ping()

redis.set(REDIS_PREFIX + "user:1", "alice")
redis.set(REDIS_PREFIX + "user:2", "bob", ex = 60)  # expire in 60s

# SETNX — set only if absent (locks, idempotent keys)
ok = redis.set(REDIS_PREFIX + "lock:job", "1", nx = True, ex = 10)
print("SETNX lock:", ok)

print("GET user:1:", redis.get(REDIS_PREFIX + "user:1"))
print("TTL user:2:", redis.ttl(REDIS_PREFIX + "user:2"))

redis.incr(REDIS_PREFIX + "counter")
redis.incrby(REDIS_PREFIX + "counter", 5)
print("INCR counter:", redis.get(REDIS_PREFIX + "counter"))

# SCAN — iterate keys matching pattern (avoid KEYS in prod)
cursor = 0
keys_sample = []
while True:
    cursor, batch = redis.scan(cursor = cursor, match = REDIS_PREFIX + "*", count = 10)
    keys_sample.extend(batch)
    if (cursor == 0): break
print("SCAN sample (first 8):", keys_sample[:8])

# Delete keys we created in this demo
if keys_sample:
    redis.delete(*keys_sample)
    print("Cleanup OK")

SETNX lock: True
GET user:1: alice
TTL user:2: 60
INCR counter: 6
SCAN sample (first 8): ['TRAINING:user:1', 'TRAINING:lock:job', 'TRAINING:counter', 'TRAINING:user:2', 'TRAINING:ks:set']
Cleanup OK


## 2. Core Redis data types

Beyond plain strings: **lists**, **hashes**, **sets**, **sorted sets** (ZSET), plus **bitmaps**, **HyperLogLog**, **streams** (later sections). Below is a **dense tour** of the most-used commands.


In [3]:
from redis import Redis

redis = Redis(**REDIS_ARGS)
redis.ping()

# --- Lists ---
redis.delete(REDIS_PREFIX + "tasks")
redis.rpush(REDIS_PREFIX + "tasks", "a", "b", "c", "to-be-trimmed")
redis.lpush(REDIS_PREFIX + "tasks", "z")
redis.ltrim(REDIS_PREFIX + "tasks", 0, 3)
print("LRANGE:", redis.lrange(REDIS_PREFIX + "tasks", 0, -1))
print("LPOP:", redis.lpop(REDIS_PREFIX + "tasks"), "RPOP:", redis.rpop(REDIS_PREFIX + "tasks"))
print("LLEN:", redis.llen(REDIS_PREFIX + "tasks"))
redis.lset(REDIS_PREFIX + "tasks", 0, "A")
print("After LSET:", redis.lrange(REDIS_PREFIX + "tasks", 0, -1))

# --- Hashes ---
redis.delete(REDIS_PREFIX + "profile:1")
redis.hset(REDIS_PREFIX + "profile:1", mapping={"name": "Ada", "role": "dev", "score": "99"})
redis.hincrby(REDIS_PREFIX + "profile:1", "score", 1)
print("HGET name:", redis.hget(REDIS_PREFIX + "profile:1", "name"))
print("HGETALL:", redis.hgetall(REDIS_PREFIX + "profile:1"))
print("HMGET:", redis.hmget(REDIS_PREFIX + "profile:1", ["name", "missing"]))

# --- Sets ---
redis.delete(REDIS_PREFIX + "tags")
redis.sadd(REDIS_PREFIX + "tags", "python", "redis", "python")
print("SMEMBERS:", sorted(redis.smembers(REDIS_PREFIX + "tags")))
redis.sadd(REDIS_PREFIX + "tags_b", "python", "go")
print("SINTER:", redis.sinter(REDIS_PREFIX + "tags", REDIS_PREFIX + "tags_b"))
print("SUNION store:", redis.sunionstore(REDIS_PREFIX + "tags_u", REDIS_PREFIX + "tags", REDIS_PREFIX + "tags_b"))

# --- Sorted sets (scores) ---
redis.delete(REDIS_PREFIX + "leaderboard")
redis.zadd(REDIS_PREFIX + "leaderboard", mapping = {
    "alice": 100, "bob": 200, "carol": 150, "david": 175, "eric": 50})
redis.zincrby(REDIS_PREFIX + "leaderboard", 10, "alice")
print("ZCARD, set size:", redis.zcard(REDIS_PREFIX + "leaderboard"))
print("ZRANGE with scores:", redis.zrange(REDIS_PREFIX + "leaderboard", 0, -1, withscores = True))
print("ZREVRANK bob:", redis.zrevrank(REDIS_PREFIX + "leaderboard", "bob"))
print("ZRANGEBYSCORE:", redis.zrangebyscore(REDIS_PREFIX + "leaderboard", 140, 210))
removed = redis.zremrangebyrank(REDIS_PREFIX + "leaderboard", 0, 2)
print("ZREMRANGEBYRANK removed:", removed)
content = redis.zrange(REDIS_PREFIX + "leaderboard", 0, 2, withscores = True)
print("ZREMRANGEBYRANK kept:", content)

# --- String/bit extras (optional) ---
redis.setbit(REDIS_PREFIX + "bits", 0, 1)
redis.setbit(REDIS_PREFIX + "bits", 7, 1)
print("GETBIT at pos 0:", redis.getbit(REDIS_PREFIX + "bits", 0))
print("BITCOUNT:", redis.bitcount(REDIS_PREFIX + "bits"))

# Cleanup
for k in [REDIS_PREFIX + "tasks", REDIS_PREFIX + "profile:1", REDIS_PREFIX + "tags", REDIS_PREFIX + "tags_b", REDIS_PREFIX + "tags_u", REDIS_PREFIX + "leaderboard", REDIS_PREFIX + "bits"]:
    redis.delete(k)
print("Data-structure demo OK")

LRANGE: ['z', 'a', 'b', 'c']
LPOP: z RPOP: c
LLEN: 2
After LSET: ['A', 'b']
HGET name: Ada
HGETALL: {'name': 'Ada', 'role': 'dev', 'score': '100'}
HMGET: ['Ada', None]
SMEMBERS: ['python', 'redis']
SINTER: {'python'}
SUNION store: 3
ZCARD, set size: 5
ZRANGE with scores: [('eric', 50.0), ('alice', 110.0), ('carol', 150.0), ('david', 175.0), ('bob', 200.0)]
ZREVRANK bob: 0
ZRANGEBYSCORE: ['carol', 'david', 'bob']
ZREMRANGEBYRANK removed: 3
ZREMRANGEBYRANK kept: [('david', 175.0), ('bob', 200.0)]
GETBIT at pos 0: 1
BITCOUNT: 2
Data-structure demo OK


## 3. PUB/SUB in one notebook

**Pub/Sub** is fire-and-forget: messages are **not persisted**—if no subscriber is connected, messages are dropped. Run the cell below: a **publisher** and **subscriber** run in **threads** for about **30 seconds** in the **same process** (no extra `.py` files).


In [4]:
## 3. PUB/SUB in one notebook

import os, threading, time, numpy
from pandas import Timestamp, Timedelta
from redis import Redis

class PubSubTest:
    KEY_PREFIX = REDIS_PREFIX.strip(":")
    def __init__(self, channel: str = "demo", wait: float = 1.0):
        self.redis_pub, self.redis_sub = Redis(**REDIS_ARGS), Redis(**REDIS_ARGS)
        self.thread_pub = threading.Thread(target = self._pub, daemon = True)
        self.thread_sub = threading.Thread(target = self._sub, daemon = True)
        self.channel = self.KEY_PREFIX + ":" + channel
        self.finished = threading.Event()
        self.counter = 0
        self.wait = wait

    @classmethod
    def print_message(cls, message: str, source: str):
        now = Timestamp.utcnow().strftime("%H:%M:%S.%f")[: -2]
        if (source == "PUB"): print(f"[{now}] {source} ==>", end = " ")
        else: print(f"{message} ==> {source} [{now}]", end = "     \n")

    def run(self, *args, **kwargs):
        self.duration = Timedelta(*args, **kwargs)
        self.finish = Timestamp.utcnow() + self.duration
        self.thread_sub.start(); self.thread_pub.start()
        while (Timestamp.utcnow() < self.finish): continue
        self.thread_pub.join(timeout = self.wait)
        self.thread_sub.join(timeout = self.wait)
        self.finished.set()
        
    def _pub(self):
        while not self.finished.is_set():
            self.counter = self.counter + 1
            now = Timestamp.utcnow()
            now_int = now.timestamp()
            rvalues = numpy.random.random(4)
            rvalues = map("{:.3f}".format, rvalues)
            message = f"{now_int * 1e6:.0f}, {self.counter}, "
            message += str.join(", ", map(str, rvalues))
            time.sleep(self.wait)
            self.redis_pub.publish(self.channel, message)
            self.print_message(message, "PUB")

    def _sub(self):
        pubsub = self.redis_sub.pubsub()
        pubsub.subscribe(self.channel)
        while not self.finished.is_set():
            message = pubsub.get_message(
                timeout = self.wait / 2)
            if message is None: continue
            if (message.get("type", None) != "message"): continue
            if (message.get("data", None) is None): continue
            time.sleep(0.001)
            self.print_message(message["data"], "SUB")

test = PubSubTest().run(seconds = 30)

[05:24:36.1488] PUB ==> 1773984274954098, 1, 0.403, 0.823, 0.917, 0.657 ==> SUB [05:24:36.2412]     
[05:24:37.2066] PUB ==> 1773984276150791, 2, 0.774, 0.891, 0.422, 0.592 ==> SUB [05:24:37.2227]     
[05:24:38.2797] PUB ==> 1773984277222621, 3, 0.226, 0.601, 0.522, 0.524 ==> SUB [05:24:38.3115]     
[05:24:39.3979] PUB ==> 1773984278311258, 4, 0.541, 0.913, 0.591, 0.034 ==> SUB [05:24:39.4298]     
[05:24:40.4746] PUB ==> 1773984279413893, 5, 0.669, 0.412, 0.748, 0.662 ==> SUB [05:24:40.5054]     
[05:24:41.5525] PUB ==> 1773984280489826, 6, 0.075, 0.995, 0.442, 0.017 ==> SUB [05:24:41.6148]     
[05:24:42.6390] PUB ==> 1773984281583405, 7, 0.906, 0.312, 0.271, 0.559 ==> SUB [05:24:42.6541]     
[05:24:43.7246] PUB ==>1773984282639234, 8, 0.383, 0.344, 0.005, 0.373 ==> SUB [05:24:43.7397]     
 [05:24:44.8138] PUB ==> 1773984283739955, 9, 0.679, 0.519, 0.527, 0.045 ==> SUB [05:24:44.9076]     
[05:24:45.8878] PUB ==> 1773984284829376, 10, 0.847, 0.753, 0.391, 0.244 ==> SUB [05:24:45.

[05:25:07.4610] PUB ==> 

## 4. Keyspace notifications

Redis can emit **events** on key changes over a **Pub/Sub channel** (`__keyevent@<db>__:*` and `__keyspace@<db>__:<key>`). You must enable **`notify-keyspace-events`** on the server (may require admin). The demo below **sets the config** when possible, then listens for **`set`** events on DB `0` for a short window.

> In managed Redis (Elasticache, etc.), config may be locked; use the provider’s UI.


In [10]:
import os, threading, time, numpy
from pandas import Timestamp, Timedelta
from redis import Redis

class PubKSNTest:
    MSG_SEP = ", "
    OP_RW = dict(set = "get", rpush = "lrange",
      sadd = "smembers", hset = "hgetall",) #zadd = "zmembers")
    KEY_PREFIX = REDIS_PREFIX.strip(":")
    def __init__(self, key_rw: str = "ks", keyspaces: list = None, wait: float = 1.0):
        self.redis_pub, self.redis_sub = Redis(**REDIS_ARGS), Redis(**REDIS_ARGS)
        self.thread_pub = threading.Thread(target = self._pub, daemon = True)
        self.thread_sub = threading.Thread(target = self._sub, daemon = True)
        self.channel = self.KEY_PREFIX + ":" + key_rw
        if (keyspaces is None): keyspaces = list(self.OP_RW.keys())
        self.keyspaces = ["__keyevent@0__:" + key for key in keyspaces]
        self.finished = threading.Event()
        self.counter = 0
        self.wait = wait

    @classmethod
    def print_message(cls, message: str, source: str):
        now = Timestamp.utcnow().strftime("%H:%M:%S.%f")[: -2]
        if (source == "PUB"): print(f"[{now}] {source} ==>", end = " ")
        else: print(f"{message} ==> {source} [{now}]", end = "     \n")

    def random_op(self, message: str):
        op_name = numpy.random.choice([*self.OP_RW])
        op = getattr(self.redis_pub, op_name, print)
        channel = self.channel + ":" + op_name
        
        if (op_name == "hset"):
            key = message[: message.find(self.MSG_SEP)]
            op(channel, key = key, value = message)
        else: op(channel, message)          
        return op_name

    def run(self, *args, **kwargs):
        parameter = "notify-keyspace-events"
        config: dict = self.redis_pub.config_get()
        prev_config_ksn = config.get(parameter, "")
        self.redis_pub.config_set(parameter, "KEA")
        self.duration = Timedelta(*args, **kwargs)
        self.finish = Timestamp.utcnow() + self.duration
        self.thread_sub.start(); self.thread_pub.start()
        while (Timestamp.utcnow() < self.finish): continue
        self.thread_pub.join(timeout = self.wait)
        self.thread_sub.join(timeout = self.wait)
        self.finished.set()
        self.redis_pub.config_set(parameter, prev_config_ksn)
        self.delete()

    def delete(self):
        pattern = self.channel + "*"
        keys = self.redis_sub.keys(pattern)
        self.redis_sub.delete(*keys)
        keys = self.redis_sub.keys(pattern)
        assert not keys, "DELETE ERROR"
        
    def _pub(self):
        while not self.finished.is_set():
            self.counter = self.counter + 1
            rvalues = numpy.random.random(4)
            n_str = str(self.counter).rjust(4, "0")
            rvalues = map("{:.3f}".format, rvalues)
            now = str(int(Timestamp.utcnow().timestamp() * 1e6))
            message = str.join(self.MSG_SEP, map(str, rvalues))
            message = str.join(self.MSG_SEP, [now, n_str, message])
            time.sleep(self.wait)
            op_name = self.random_op(message)
            message = f" <{op_name}> {message}"  
            self.print_message(message, "PUB")

    def _sub(self):
        pubsub = self.redis_sub.pubsub()
        for keyspace in self.keyspaces:
            pubsub.psubscribe(keyspace)
        timeout = timeout = self.wait / 2
        while not self.finished.is_set():
            pmessage = pubsub.get_message(timeout = timeout)
            if (pmessage is None) or not pmessage: continue
            if (pmessage.get("type") != "pmessage"): continue
            if (pmessage.get("data", None) is None): continue
            time.sleep(0.001)
            key = str(pmessage["data"])
            op_name_pub = str.split(key, ":")[-1]
            op_name = self.OP_RW.get(op_name_pub)
            op = getattr(self.redis_sub, op_name)
            if (op_name in {"lrange"}):
                message = op(key, -1, -1)
            else: message = op(key)
            message = f" <{op_name}> {message}"
            self.print_message(message, "SUB")

test = PubKSNTest()
test.run(seconds = 30)

[06:38:18.7183] PUB ==>  <get> 1773988697642157, 0001, 0.308, 0.779, 0.711, 0.734 ==> SUB [06:38:18.8892]     
 <lrange> ['1773988698749749, 0002, 0.059, 0.477, 0.168, 0.428'] ==> SUB [06:38:19.8265]     
[06:38:19.8269] PUB ==> [06:38:20.9457] PUB ==>  <smembers> {'1773988699826997, 0003, 0.544, 0.970, 0.401, 0.299'} ==> SUB [06:38:21.0082]     
[06:38:21.9914] PUB ==>  <smembers> {'1773988700961558, 0004, 0.615, 0.050, 0.671, 0.670', '1773988699826997, 0003, 0.544, 0.970, 0.401, 0.299'} ==> SUB [06:38:22.0380]     
[06:38:23.1294] PUB ==> <smembers> {'1773988702007541, 0005, 0.748, 0.894, 0.015, 0.628', '1773988700961558, 0004, 0.615, 0.050, 0.671, 0.670', '1773988699826997, 0003, 0.544, 0.970, 0.401, 0.299'} ==> SUB [06:38:23.1603]     
 [06:38:24.2523] PUB ==>  <lrange> ['1773988703176081, 0006, 0.859, 0.584, 0.772, 0.854'] ==> SUB [06:38:24.2682]     
[06:38:25.3451] PUB ==>  <smembers> {'1773988702007541, 0005, 0.748, 0.894, 0.015, 0.628', '1773988700961558, 0004, 0.615, 0.050, 0

[06:38:50.5132] PUB ==> 

## 5. Streams: fundamentals

**Streams** are append-only logs identified by keys. Entries have **IDs** `millisecondsSequence` (e.g. `1700000000000-0`). Core commands: **`XADD`**, **`XRANGE`/`XREVRANGE`**, **`XREAD`** (blocking optional), **`XLEN`**.

Unlike Pub/Sub, stream entries are **stored** until trimmed or expired (TTL on the key if set).


In [ ]:
from redis import Redis

STREAM = REDIS_PREFIX + ":test_stream"

redis = Redis(**REDIS_ARGS)
redis.ping()
redis.delete(STREAM)

eid1 = redis.xadd(STREAM, {"sku": "A1", "qty": "2"})
eid2 = redis.xadd(STREAM, {"sku": "B2", "qty": "1"})
print("XADD ids:", eid1, eid2)
print("XLEN:", redis.xlen(STREAM))

print("XRANGE all:", redis.xrange(STREAM, min="-", max="+"))

# XREAD from beginning
block_ms = 1000
out = redis.xread({STREAM: "0-0"}, count=10, block=block_ms)
print("XREAD:", out)

redis.delete(STREAM)
print("Streams fundamentals OK")

XADD ids: 1773989238736-0 1773989238737-0
XLEN: 2
XRANGE all: [('1773989238736-0', {'sku': 'A1', 'qty': '2'}), ('1773989238737-0', {'sku': 'B2', 'qty': '1'})]
XREAD: [['TRAINING::test_stream', [('1773989238736-0', {'sku': 'A1', 'qty': '2'}), ('1773989238737-0', {'sku': 'B2', 'qty': '1'})]]]
Streams fundamentals OK


## 6. Streams: consumer groups

**Consumer groups** partition work: each message is delivered to **one** consumer in the group (roughly). Flow:

1. **`XGROUP CREATE stream group id`** — start from `$` (new only) or `0` (replay from start).
2. **`XREADGROUP GROUP g consumer BLOCK … STREAM key >`** — read **new** pending-to-group entries.
3. **`XACK stream group id`** — mark as processed.

Pending entries (not ACKed) live in a **PEL** (Pending Entries List) inspectable with **`XPENDING`**.


In [27]:
from redis import Redis

STREAM = REDIS_PREFIX + ":test_stream"
GROUP = "workers"
CONSUMER = "c1"

redis = Redis(**REDIS_ARGS)
redis.ping()
redis.delete(STREAM)

redis.xadd(STREAM, {"task": "t1"})
redis.xadd(STREAM, {"task": "t2"})

try:
    redis.xgroup_create(STREAM, GROUP, id = "0", mkstream = True)
except redis.ResponseError as e:
    if "BUSYGROUP" not in str(e):
        raise

# Read two messages as consumer c1
resp = redis.xreadgroup(GROUP, CONSUMER, {STREAM: ">"}, count = 2, block = 1000)
print("XREADGROUP:", resp)

ids = [entry[0] for _sn, entries in resp for entry in entries]
for eid in ids: redis.xack(STREAM, GROUP, eid)
print("XPENDING after ACK:", redis.xpending(STREAM, GROUP))

XREADGROUP: [['TRAINING::test_stream', [('1774000640417-0', {'task': 't1'}), ('1774000640418-0', {'task': 't2'})]]]
XPENDING after ACK: {'pending': 0, 'min': None, 'max': None, 'consumers': []}


## 7. Streams: multi-consumer patterns

**Horizontal scale:** multiple processes use the **same group** with **different consumer names**—Redis routes messages so each entry is claimed by **one** consumer (when using `>` reads).

**Recovery:** if a consumer dies, entries stay **pending**. Use **`XPENDING`** / **`XCLAIM`** / **`XAUTOCLAIM`** to **reassign** stale messages. Below: two consumers, one message left unacked, then **`XAUTOCLAIM`** moves it.


In [ ]:
import os, time
from redis import Redis

STREAM = "g_redis:stream:mc"
GROUP = "g"

redis = Redis(**REDIS_ARGS)
redis.ping()
redis.delete(STREAM)

eid = redis.xadd(STREAM, {"k": "v"})
redis.xgroup_create(STREAM, GROUP, id = "0", mkstream = True)

# c1 reads but does NOT ack — simulates crash
redis.xreadgroup(GROUP, "c1", {STREAM: ">"}, count = 1, block = 500)
pend = redis.xpending(STREAM, GROUP)
print("XPENDING summary:", pend)

# Autoclaim messages idle for >= 0 ms (Redis 6.2+)
time.sleep(0.2)
ac = redis.xautoclaim(STREAM, GROUP, "c2", min_idle_time = 0, start_id = "0-0", count = 10)
print("XAUTOCLAIM:", ac)

# ACK from c2
redis.xack(STREAM, GROUP, eid)
print("XPENDING after ack:", redis.xpending(STREAM, GROUP))

redis.delete(STREAM)
print("Multi-consumer / recovery demo OK")

XPENDING summary: {'pending': 1, 'min': '1774000435149-0', 'max': '1774000435149-0', 'consumers': [{'name': 'c1', 'pending': 1}]}
XAUTOCLAIM: ['0-0', [('1774000435149-0', {'k': 'v'})], []]
XPENDING after ack: {'pending': 0, 'min': None, 'max': None, 'consumers': []}
Multi-consumer / recovery demo OK


## 8. Streams: persistence and trimming

**Message persistence** is really two layers:

- **Process safety:** once **`XADD`** returns, the entry is in Redis memory (and will be written according to **AOF/RDB** policy—see next section).
- **Bounded memory:** use **`XADD … MAXLEN ~ N`** or **`XTRIM`** to cap stream length. The `~` is **approximate** trimming (more efficient).

Setting **`EXPIRE`** on the stream key TTL-evicts the **whole** stream.

### Trimming vs consumers

**`MAXLEN` / `XTRIM` drop the oldest entries** in the stream. If producers are fast and consumers are slow, entries can be **removed before** a consumer ever **`XREADGROUP`**’s them—those messages are **gone** from Redis (this is separate from **ACK** semantics). Size the buffer (`N`) and consumer throughput accordingly.

**`DEL` on the stream key** removes the stream **and** all **consumer-group** metadata on that key. After a delete you need **`XGROUP CREATE`** again (and consumers may see **`NOGROUP`** until you do).

### `XGROUP CREATE` and the starting `id` (backlog vs live-only)

When you create a group, the **`id`** argument sets where that group **starts** in the stream timeline:

- **`id="$"`** (the default in **redis-py** if you omit `id`): “**Only messages added after the group is created.**” Anything already in the stream at create time is **skipped** for reads that use **`>`** (new deliveries to the group). Use this when you want a **fresh** consumer that must **not** replay history.
- **`id="0"` / `"0-0"`**: “**Start from the beginning of the stream.**” The group has not delivered anything yet, so **`XREADGROUP … STREAMS mystream ">"`** will hand you **existing** entries first (in order), then every later **`XADD`**. Use this when each consumer must see the **backlog at least once** (typical when producers may **`XADD` before** consumers run **`XGROUP CREATE`**).

**`>`** means: entries **not yet delivered** to this group (not “globally new to Redis”). It is unrelated to Pub/Sub; there is no **`.listen()`**—you **`while` loop** with **`XREADGROUP`** and **`block=…` ms** (returns early when data arrives, or after the timeout with an empty result).

**`mkstream=True`:** if the stream key **does not exist**, create an **empty** stream so **`XGROUP CREATE`** can succeed without a prior **`XADD`**.

### `XREADGROUP` usage (quick recap)

- **Signature (redis-py):** `xreadgroup(groupname, consumername, streams, count=None, block=None, noack=False)` with **`streams`** a dict **`{ stream_key: id, ... }`**. For **only undelivered** messages to the group, use **`">"`** for each stream.
- **One call, several streams:** you can block on **multiple** keys; **`block=100`** waits **up to** 100 ms for **any** of them to have work, then returns (possibly empty).
- **Each stream** must already have **`XGROUP CREATE`** for that **`groupname`** (group state is **per stream key**; the same group **name** on two keys is two independent groups).
- After processing, **`XACK`** removes the entry from the **PEL** (pending list); without **`NOACK`**, unacked messages stay pending for **retry / `XCLAIM` / `XAUTOCLAIM`**.

**Threading note:** exceptions inside **`ThreadPoolExecutor`** workers are **silent** until you call **`.result()`** on the returned **Future**—if consumers “print nothing,” verify **`NOGROUP`**, wrong **`id`**, or races with **`DEL`** on the stream.


In [122]:
from redis import Redis
import time, numpy, pandas, tqdm
from concurrent.futures import ThreadPoolExecutor

class StreamTest:

    MAX_LEN, SEED_MAX_EXP = 10, 6
    STREAM_PREFIX = REDIS_PREFIX.strip(":")
    def __init__(self, network: dict):

        self.network = network.copy()
        producers, consumers = set(), set(network)
        for prod in network.values(): producers.update(prod)
        self.producers = {key: Redis(**REDIS_ARGS) for key in producers}
        self.consumers = {key: Redis(**REDIS_ARGS) for key in consumers}

    @classmethod
    def print_message(cls, message: dict, source: str, prod: bool):

        arrow = "==>" if prod else "<=="
        now = pandas.Timestamp.utcnow().strftime("%H:%M:%S.%f")[: -2]
        print(f"[{now} | {source} {arrow}] {message}")

    def run(self, mps: int = 100, seconds: int = 30):

        self.reset()
        self.shutdown = False
        self.data = dict()
        pool = ThreadPoolExecutor(4)
        self.idle = len(self.producers) / mps
        {key: pool.submit(self._cons, cons, key)
          for key, cons in self.consumers.items()}
        {key: pool.submit(self._prod, prod, key)
          for key, prod in self.producers.items()}

        steps = 2000
        iterator = tqdm.tqdm(range(steps))
        for step in iterator:
            legend = f"Entries: {len(self.data)}"
            iterator.set_description(legend)
            time.sleep(seconds / steps)
        self.shutdown = True
        pool.shutdown(wait = 1)
        self.reset()

        df = pandas.DataFrame.from_dict(self.data, orient = "index")
        df = df.rename_axis(["mid", "key", "ts"]).sort_index()
        return df.rename_axis("fields", axis = "columns")

    def reset(self):

        for key, prod in self.producers.items():
            stream = self.STREAM_PREFIX + ":" + key
            print(f"WARNING: Deleted stream \"{stream}\"")
            prod.delete(stream)

    def _seed(self):

        seed_p = numpy.random.random() * self.SEED_MAX_EXP
        seed_p = (seed_p - 0.6) / 2
        seed_q = self.SEED_MAX_EXP - seed_p + 1
        digits = self.SEED_MAX_EXP - int(seed_p)
        price = round(10.0 ** seed_p, digits)
        qty = int(10.0 ** seed_q)
        return price, digits, qty

    def _prod(self, redis: Redis, prod: str):

        try:
            stream = self.STREAM_PREFIX + ":" + prod
            bid, digits, qmax = self._seed()
            tick = pow(10, - digits)

            while not self.shutdown:
                randoms = numpy.random.random(5)
                p_tick_sign = numpy.sign(randoms[0] * 2 - 1)
                v_side_diff = 1 + 0.1 * (randoms[1] * 2 - 1)
                spread, qty = 10 * randoms[2], qmax * randoms[3]
                bid = round(bid + tick * p_tick_sign, digits)
                ask = round(bid + tick * spread, digits)
                time.sleep((randoms[4] * 0.2 + 0.4) * self.idle)
                now = pandas.Timestamp.utcnow()
                ts = int(now.timestamp() * 1e6)
                message = {"t": ts, "pa": float(ask), "pb": float(bid),
                          "qa": int(v_side_diff * qty), "qb": int(qty)}                
                mid = redis.xadd(stream, fields = message,
                    maxlen = self.MAX_LEN, approximate = True)
                #self.print_message(message, prod, prod = True)
                self.data[(mid, prod, now)] = message

        except Exception as EXC: print(prod, "@", repr(EXC))

    def _cons(self, redis: Redis, cons: str):

        try:
            group = self.STREAM_PREFIX + ":" + cons
            streams = dict()
            link = (group, cons, streams)
            for prod in self.network[cons]:
                stream = self.STREAM_PREFIX + ":" + prod
                redis.xgroup_create(stream, group, "0", mkstream = True)
                streams[stream] = ">"
            while not self.shutdown:
                messages = redis.xreadgroup(*link, block = 100)
                if (messages is None) or not messages: continue
                stream, messages = messages[0]
                prod = stream.split(":")[-1]
                for mid, message in messages:
                    now = pandas.Timestamp.utcnow()
                    message = {"str": prod, **message}
                    self.data[(mid, cons, now)] = message
                    #self.print_message(message, cons, prod = False)

        except Exception as EXC: print(cons, "@", repr(EXC))

In [123]:
network = {
    "cons_1": {"prod_1", "prod_2"},
    "cons_2": {"prod_2"}
}
test = StreamTest(network)
data = test.run(200, seconds = 30)

dc: pandas.DataFrame = data.loc[data["str"].notna()]
dp: pandas.DataFrame = data.loc[data["str"].isna()]
dp = dp.reset_index(["key", "ts"]).rename(columns = {"key": "prod"})
dc = dc.reset_index(["key", "ts"]).rename(columns = {"key": "cons"})
dc = dc.rename(columns = {"str": "prod"}); dp.pop("str")
dp = dp.set_index("prod", append = True)
dc = dc.set_index("prod", append = True)
df = pandas.DataFrame.merge(self = dc, right = dp, how = "inner",
    left_index = True, right_index = True, suffixes = ["c", "p"])
df = df[["cons", "tsc", "tsp"]]
df["delay"] = df.pop("tsc") - df["tsp"]
df["delay"] = df["delay"].dt.total_seconds()
df["delay"] = (df["delay"] * 1e6).astype(int)
df["delay"].describe()

Entries: 56381: 100%|██████████| 2000/2000 [00:44<00:00, 45.25it/s]


count     11340.000000
mean       4762.337302
std       16222.061980
min         335.000000
25%         817.000000
50%        1165.000000
75%        2888.250000
max      292362.000000
Name: delay, dtype: float64

## 9. Pipelines, transactions (`MULTI`/`EXEC`), and Lua

- **Pipeline:** batch commands to reduce **round-trips** (not atomic by default).
- **`MULTI`/`EXEC`:** **atomic** batch; conflicts fail the transaction in typical use with **`WATCH`**.
- **Lua (`EVAL`):** **atomic** server-side scripts—great for **conditional** updates (e.g. compare-and-swap) with a single round-trip.


In [24]:
from redis import Redis

redis = Redis(**REDIS_ARGS)

redis.ping()
k = REDIS_PREFIX + ":pipeline_test"
redis.delete(k)
redis.set(k, "10")

pipe_1 = redis.pipeline()
pipe_1.incrby(k, 5)
pipe_1.get(k)
print("Pipeline 1:", pipe_1.execute())

pipe_2 = redis.pipeline()
pipe_2.multi()
pipe_2.incrby(k, 1)
pipe_2.get(k)
print("Pipeline 2:", pipe_2.execute())

# Lua: return new value only if old == expected
lua_script = """
local cur = redis.call('GET', KEYS[1])
if cur == ARGV[1] then
    redis.call('SET', KEYS[1], ARGV[2])
    return {ok='SET', new=ARGV[2]}
end
return {err='CAS_FAILED', cur=cur}
"""
res = redis.eval(lua_script, 1, k, "16", "99")
print("EVAL CAS:", res)
redis.delete(k);

Pipeline 1: [15, '15']
Pipeline 2: [16, '16']
EVAL CAS: SET


## 10. RDB, AOF, and replication (overview)

- **RDB:** point-in-time snapshots (compact, can lose last minutes if only RDB).
- **AOF:** append log of writes; **fsync policy** trades durability vs throughput (`always` / `everysec` / `no`).
- **Replication:** primary/replica async replication; **sentinel** or **Cluster** for HA—client libraries expose different connection modes.

These are **server** settings—not something you set per Python call—but they define **how durable** your stream messages really are after `XADD` succeeds.


## 11. Practical checklist

- Prefer **`SCAN`** over **`KEYS`** in production.
- Use **TTL** or **`MAXLEN`** on streams to avoid unbounded RAM.
- For work queues at scale: **consumer groups** + **`XACK`** + **`XAUTOCLAIM`** for poison / stale messages.
- Remember **Pub/Sub is not durable**; use **Streams** (or an external broker) when you need replay and retention.
- Load-test **`BLOCK`** timeouts and **consumer count**—one slow consumer can grow **pending** lists.
- In **managed Redis**, verify **`notify-keyspace-events`** and **`XGROUP`** capabilities match your plan.


## 12. Companion: trading and low-latency patterns

This notebook is a **reference** for Redis commands and semantics. The companion **[`g_redis-hft.ipynb`](g_redis-hft.ipynb)** shows **how** those pieces fit **distributed trading**: ingesting **ticks**, **book snapshots**, **candles**, and **alt data** from **many exchanges**, with **bounded memory** and **predictable recovery**.

The **logic** is **separation of concerns**. Connectors emit **high-volume**, **duplicate** events; downstream needs **replay**, **deduplication**, and **shared** state across services. **Streams** and **consumer groups** buffer **multi-consumer** pipelines; **sorted sets** give **time-scored** windows; **hashes** hold **latest** snapshots; **`SET NX`** backs **idempotency** on retries. **Pub/Sub** fits only where **loss** is acceptable. **Lua** can atomically check **risk** before external placement calls.

**Why** a second notebook: **domain context**—Redis as a **µs–ms** operational layer (not a nanosecond matcher), **sharding** by symbol or venue, and **source of truth**: **orders**/**fills** stay in **databases** and at the **exchange**; Redis speeds **coordination** and **streaming**, not regulatory records.

**Continue:** **[`g_redis-hft.ipynb`](g_redis-hft.ipynb)** — *Redis in low-latency & distributed trading (Python)*.
